# FDTE TinyML - Reproducible Computational Proof of Concept

Supplementary notebook for the accepted Latin.Science 2026 paper. The notebook executes the same canonical pipeline implemented in `src/fdte_experiment.py`.


## 1. Environment and repository paths

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import sklearn
import tensorflow as tf

def find_root():
    start = Path.cwd().resolve()
    candidates = [start, *start.parents]
    content = Path('/content')
    if content.exists():
        candidates.extend([content, *[p for p in content.iterdir() if p.is_dir()]])
    for candidate in candidates:
        if (candidate / 'data' / 'pm25').exists() and (candidate / 'src' / 'fdte_experiment.py').exists():
            return candidate
    raise FileNotFoundError(
        'Repository root not found. Clone or upload/extract the complete repository, preserving its folders.'
    )

ROOT = find_root()
sys.path.insert(0, str(ROOT))
from src import fdte_experiment as exp

print('Repository root:', ROOT)
print('Python:', sys.version.split()[0])
print('TensorFlow:', tf.__version__)
print('NumPy:', np.__version__)
print('pandas:', pd.__version__)
print('scikit-learn:', sklearn.__version__)

## 2. Input-data audit

In [ ]:
PM25_DIR = ROOT / 'data' / 'pm25'
for country, filename in exp.COUNTRY_FILES.items():
    df = exp.load_country_snapshot(PM25_DIR, filename)
    pollutant = df['poluente'].astype(str).str.lower().str.replace('.', '', regex=False)
    pm25 = df[pollutant.eq('pm25')]
    print(
        f'{country}: {len(df):,} study-snapshot records; '
        f'{len(pm25):,} PM2.5 records; {pm25.estacao.nunique()} PM2.5 stations'
    )


## 3. Execute the canonical pipeline

This cell regenerates:

- the transparent contextual z-score summary;
- chronological station-level train/validation/test splits;
- the three PM2.5 predictive neural models;
- full-integer INT8 TFLite exports;
- predictive metrics and station audits;
- FDTE weight sensitivity and TCO sensitivity results;
- preprocessing and model metadata.


In [ ]:
exp.main()

## 4. Reported predictive results

In [ ]:
performance = pd.read_csv(ROOT / 'results' / 'model_performance_temporal.csv')
performance

## 5. Contextual rule and sensitivity analyses

In [ ]:
contextual = pd.read_csv(ROOT / 'results' / 'contextual_zscore_summary.csv')
weights = pd.read_csv(ROOT / 'results' / 'fdte_weight_sensitivity.csv')
tco = pd.read_csv(ROOT / 'results' / 'tco_sensitivity.csv')
print('Contextual z-score summary')
display(contextual)
print('FDTE weight sensitivity')
display(weights)
print('TCO sensitivity')
display(tco)

## 6. Validate the generated TFLite files

In [ ]:
for country, slug in exp.COUNTRY_SLUG.items():
    model_path = ROOT / 'models' / f'{slug}_pm25_prediction_3h.tflite'
    interpreter = tf.lite.Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()
    input_detail = interpreter.get_input_details()[0]
    output_detail = interpreter.get_output_details()[0]
    print(
        country,
        'size=', model_path.stat().st_size, 'bytes;',
        'input=', input_detail['shape'].tolist(), input_detail['dtype'].__name__,
        'output=', output_detail['shape'].tolist(), output_detail['dtype'].__name__,
    )

## 7. Validity notes

- The 25 µg/m³ threshold is operational and does not replace the WHO 24-hour averaging period.
- OpenAQ coverage is limited to the stations and periods available in the included snapshots.
- The repository distributes the exact PM2.5 input snapshots used by the computational pipeline. The broader 64,734-record study inventory and two non-PM2.5 quality-control exclusions are documented in `data/pollutant_inventory.csv` and `data/quality_exclusions.csv`.
- File size and successful TFLite loading are not complete ESP32 deployment tests.
- The multigas experiment is intentionally excluded because simultaneous pollutant coverage was insufficient across the three country datasets.
- Small floating-point display differences may occur across platforms; the pinned environment in `requirements.txt` is the verified reference environment.
